# triangle-splatting 2 :: Custom data :: Celebrity_Edge_GradPlaza

-----
- Conda env : [triangle_splatting2](README.md#setup-a-conda-environment)
-----

### Check system

In [1]:
!nvidia-smi

Wed Oct 15 14:04:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 30%   44C    P8             33W /  250W |     528MiB /  11264MiB |     38%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download a video

In [2]:
import os
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)

In [3]:
import gdown

VIDEO_NAME = "Celebrity_Edge_GradPlaza"

FPS = 10
RES = 4
id = "1T6naoj4bNRZOdoKjLmI3WAd1cp-NBCTR"
vid_path = f"./temp_data/{VIDEO_NAME}.mov"


if not os.path.exists(vid_path):
    gdown.download(id=id, output = vid_path)
else:
    print(f"{vid_path} already is downloaded")

./temp_data/Celebrity_Edge_GradPlaza.mov already is downloaded


### Extract images from the video

In [4]:
DATASET_DIR_PATH = f"./temp_data/{VIDEO_NAME}"
IMAGES_DIR_PATH = os.path.join(DATASET_DIR_PATH, "images")
DATABASE_PATH = os.path.join(DATASET_DIR_PATH, "database.db")

if not os.path.exists(IMAGES_DIR_PATH):
    Path(IMAGES_DIR_PATH).mkdir(exist_ok=True, parents=True)
    !ffmpeg -i $vid_path -vf fps=$FPS $IMAGES_DIR_PATH/frame_%04d.jpg
else:
    print(f"{IMAGES_DIR_PATH} already is extracted")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Colmap :: Feature Extraction & Matching

In [5]:
SPARSE_DIR = os.path.join(DATASET_DIR_PATH, "sparse")
Path(SPARSE_DIR).mkdir(exist_ok=True, parents=True)

if not os.path.exists(DATABASE_PATH):
    # Colmap Feature Extraction
    !colmap feature_extractor \
        --database_path $DATABASE_PATH \
        --image_path $IMAGES_DIR_PATH  --ImageReader.camera_model PINHOLE
    !colmap sequential_matcher \
        --database_path $DATABASE_PATH
    !colmap mapper \
        --database_path $DATABASE_PATH \
        --image_path $IMAGES_DIR_PATH \
        --output_path $SPARSE_DIR
else:
    print(f"{DATABASE_PATH} already is extracted and reconstructed")


Feature extraction

Processed file [1/252]
  Name:            frame_0001.jpg
  Dimensions:      1080 x 1920
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        4941
Processed file [2/252]
  Name:            frame_0002.jpg
  Dimensions:      1080 x 1920
  Camera:          #2 - PINHOLE
  Focal Length:    2304.00px
  Features:        4908
Processed file [3/252]
  Name:            frame_0003.jpg
  Dimensions:      1080 x 1920
  Camera:          #3 - PINHOLE
  Focal Length:    2304.00px
  Features:        4654
Processed file [4/252]
  Name:            frame_0004.jpg
  Dimensions:      1080 x 1920
  Camera:          #4 - PINHOLE
  Focal Length:    2304.00px
  Features:        4756
Processed file [5/252]
  Name:            frame_0005.jpg
  Dimensions:      1080 x 1920
  Camera:          #5 - PINHOLE
  Focal Length:    2304.00px
  Features:        4841
Processed file [6/252]
  Name:            frame_0006.jpg
  Dimensions:      1080 x 1920
  Camera:          #6 - PI

### Triangle-Splatting :: Training (Outdoor mode)

In [6]:
DATASET_DIR_PATH
OUTDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_outdoor"
print(DATASET_DIR_PATH)
print(OUTDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/train.py -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --eval


./temp_data/Celebrity_Edge_GradPlaza
./temp_result/Celebrity_Edge_GradPlaza_outdoor
Optimizing ./temp_result/Celebrity_Edge_GradPlaza_outdoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packa

### Triangle-Splatting :: Rendering (Outdoor mode)

In [7]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/render.py -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES -s $DATASET_DIR_PATH

Looking for config file in ./temp_result/Celebrity_Edge_GradPlaza_outdoor/cfg_args
Config file found: ./temp_result/Celebrity_Edge_GradPlaza_outdoor/cfg_args
Rendering ./temp_result/Celebrity_Edge_GradPlaza_outdoor
Traceback (most recent call last):
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/render.py", line 81, in <module>
    render_sets(model.extract(args), args.iteration, pipeline.extract(args), args.skip_train, args.skip_test)
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/render.py", line 50, in render_sets
    scene = Scene(args=dataset,
            ^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/scene/__init__.py", line 44, in __init__
    self.loaded_iter = searchForMaxIteration(os.path.join(self.model_path, "point_cloud"))
               

### Triangle-Splatting :: Create a video (Outdoor mode)

In [8]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_video.py  -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --save_as $OUTDOOR_OUTPUR_DIR_PATH/output_video

Looking for config file in ./temp_result/Celebrity_Edge_GradPlaza_outdoor/cfg_args
Config file found: ./temp_result/Celebrity_Edge_GradPlaza_outdoor/cfg_args
Creating video for ./temp_result/Celebrity_Edge_GradPlaza_outdoor
Traceback (most recent call last):
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_video.py", line 44, in <module>
    scene = Scene(args=dataset,
            ^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/scene/__init__.py", line 44, in __init__
    self.loaded_iter = searchForMaxIteration(os.path.join(self.model_path, "point_cloud"))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/utils/system_utils.py", line 36, in searchForMaxIterati

In [9]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_ply.py $OUTDOOR_OUTPUR_DIR_PATH/point_cloud/iteration_30000 --out $OUTDOOR_OUTPUR_DIR_PATH/mesh.ply

Error: Could not find './temp_result/Celebrity_Edge_GradPlaza_outdoor/point_cloud/iteration_30000/point_cloud_state_dict.pt'
Traceback (most recent call last):
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_ply.py", line 80, in <module>
    main()
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_ply.py", line 60, in main
    verts, faces, f_dc, f_rest, act_deg = load_scene(args.scene_dir, device=device)
                                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_ply.py", line 18, in load_scene
    raise FileNotFoundError(f"Could not find '{path}'")
FileNotFoundError: Could not find './temp_result/Celebrity_Edge_GradPlaza_outdoor/point_cloud/iteration_30000/point_cloud_state_dict.pt'

### Triangle-Splatting :: Training (Indoor mode)

In [10]:
DATASET_DIR_PATH
INDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_indoor"
print(DATASET_DIR_PATH)
print(INDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/train.py -s $DATASET_DIR_PATH -m $INDOOR_OUTPUR_DIR_PATH -r $RES --eval  --indoor 

./temp_data/Celebrity_Edge_GradPlaza
./temp_result/Celebrity_Edge_GradPlaza_indoor
Optimizing ./temp_result/Celebrity_Edge_GradPlaza_indoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-package

### Triangle-Splatting :: Rendering (Indoor mode)

In [11]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/render.py -m $INDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Celebrity_Edge_GradPlaza_indoor/cfg_args
Config file found: ./temp_result/Celebrity_Edge_GradPlaza_indoor/cfg_args
Rendering ./temp_result/Celebrity_Edge_GradPlaza_indoor
Traceback (most recent call last):
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/render.py", line 81, in <module>
    render_sets(model.extract(args), args.iteration, pipeline.extract(args), args.skip_train, args.skip_test)
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/render.py", line 50, in render_sets
    scene = Scene(args=dataset,
            ^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/scene/__init__.py", line 44, in __init__
    self.loaded_iter = searchForMaxIteration(os.path.join(self.model_path, "point_cloud"))
                  

### Triangle-Splatting :: Create a video (Indoor mode)

In [12]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_video.py -s $DATASET_DIR_PATH -m $INDOOR_OUTPUR_DIR_PATH -r $RES --save_as $INDOOR_OUTPUR_DIR_PATH/output_video

Looking for config file in ./temp_result/Celebrity_Edge_GradPlaza_indoor/cfg_args
Config file found: ./temp_result/Celebrity_Edge_GradPlaza_indoor/cfg_args
Creating video for ./temp_result/Celebrity_Edge_GradPlaza_indoor
Traceback (most recent call last):
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_video.py", line 44, in <module>
    scene = Scene(args=dataset,
            ^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/scene/__init__.py", line 44, in __init__
    self.loaded_iter = searchForMaxIteration(os.path.join(self.model_path, "point_cloud"))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/utils/system_utils.py", line 36, in searchForMaxIteration


In [13]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_ply.py $INDOOR_OUTPUR_DIR_PATH/point_cloud/iteration_30000 --out $INDOOR_OUTPUR_DIR_PATH/mesh.ply

Error: Could not find './temp_result/Celebrity_Edge_GradPlaza_indoor/point_cloud/iteration_30000/point_cloud_state_dict.pt'
Traceback (most recent call last):
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_ply.py", line 80, in <module>
    main()
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_ply.py", line 60, in main
    verts, faces, f_dc, f_rest, act_deg = load_scene(args.scene_dir, device=device)
                                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/Triangle_splatting2/temp_triangle-splatting2/create_ply.py", line 18, in load_scene
    raise FileNotFoundError(f"Could not find '{path}'")
FileNotFoundError: Could not find './temp_result/Celebrity_Edge_GradPlaza_indoor/point_cloud/iteration_30000/point_cloud_state_dict.pt'
